# Stacking Ensemble — RoBERTa-base + DeBERTa-v3-base

combina as predições (probabilidades softmax) de dois modelos transformer num meta-classificador.

In [1]:
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    DebertaV2Tokenizer, DebertaV2ForSequenceClassification
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import cross_val_score
from transformer_utils import InferenceDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
ID2LABEL_OUT = {0: 'Google', 1: 'Anthropic', 2: 'Meta', 3: 'OpenAI', 4: 'Human'}

device: cuda


## *carregar modelos*

In [2]:
# RoBERTa-base
roberta_path = '../models/model_roberta'
roberta_tok  = RobertaTokenizer.from_pretrained(roberta_path)
roberta_model = RobertaForSequenceClassification.from_pretrained(roberta_path).to(device)
roberta_model.eval()
print('RoBERTa-base carregado')

# DeBERTa-v3-base
deberta_path = '../models/model_deberta'
deberta_tok  = DebertaV2Tokenizer.from_pretrained(deberta_path)
deberta_model = DebertaV2ForSequenceClassification.from_pretrained(deberta_path).float().to(device)
deberta_model.eval()
print('DeBERTa-v3-base carregado')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa-base carregado


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DeBERTa-v3-base carregado


## *função para extrair probabilidades*

In [3]:
MAX_LEN = 128

def get_probabilities(model, tokenizer, texts, batch_size=32):
    """Extrai probabilidades softmax de um modelo para uma lista de textos."""
    ds = InferenceDataset(texts, tokenizer, MAX_LEN)
    loader = DataLoader(ds, batch_size=batch_size)
    
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs  = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    
    return np.vstack(all_probs)

print('função definida')

função definida


## *dados para o meta-classificador*


In [4]:
# carregar dados reais
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv', sep=';')
df_subm2 = pd.read_csv('../Subm2/subm2_labels_revealed.csv', sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',    sep=';')

# combinar subm1 + subm2 para treino do meta-classificador
df_meta_train = pd.concat([df_subm1, df_subm2], ignore_index=True)

meta_train_texts = df_meta_train['Text'].fillna('').tolist()
meta_train_y     = np.array([LABEL2ID[l.strip().lower()] for l in df_meta_train['Label']])

val_texts = df_ex['Text'].fillna('').tolist()
val_y     = np.array([LABEL2ID[l.strip().lower()] for l in df_ex['Label']])

print(f'meta-train: {len(meta_train_texts)} exemplos (Subm1: {len(df_subm1)} + Subm2: {len(df_subm2)})')
print(f'validação:   {len(val_texts)} exemplos (dataset-exemplos)')
print(f'\ndistribuição meta-train:')
print(pd.Series(meta_train_y).map(ID2LABEL).value_counts())

meta-train: 200 exemplos (Subm1: 100 + Subm2: 100)
validação:   125 exemplos (dataset-exemplos)

distribuição meta-train:
human        68
google       35
meta         34
anthropic    33
openai       30
Name: count, dtype: int64


## *extrair probabilidades*

In [5]:
print('a extrair probabilidades do RoBERTa...')
probs_roberta_train = get_probabilities(roberta_model, roberta_tok, meta_train_texts)
probs_roberta_val   = get_probabilities(roberta_model, roberta_tok, val_texts)

print('a extrair probabilidades do DeBERTa...')
probs_deberta_train = get_probabilities(deberta_model, deberta_tok, meta_train_texts)
probs_deberta_val   = get_probabilities(deberta_model, deberta_tok, val_texts)

# concatenar: [roberta_5_probs, deberta_5_probs] = 10 features
X_meta_train = np.hstack([probs_roberta_train, probs_deberta_train])
X_meta_val   = np.hstack([probs_roberta_val,   probs_deberta_val])

print(f'\nfeatures meta-train: {X_meta_train.shape}')
print(f'features validação:  {X_meta_val.shape}')

a extrair probabilidades do RoBERTa...
a extrair probabilidades do DeBERTa...

features meta-train: (200, 10)
features validação:  (125, 10)


## *treinar meta-classificador*

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

classifiers = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Logistic Reg (C=10)':   LogisticRegression(max_iter=1000, C=10.0, random_state=42),
    'SVM (RBF)':             SVC(kernel='rbf', C=10, probability=True, random_state=42),
    'SVM (Linear)':          SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    'KNN (k=5)':             KNeighborsClassifier(n_neighbors=5),
    'KNN (k=3)':             KNeighborsClassifier(n_neighbors=3),
    'MLP':                   MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

print(f'{"Classificador":<25} {"CV F1-macro":>12} {"Val Acc":>10} {"Val F1":>10}')
print('-' * 60)

best_f1 = 0
best_name = ''
best_clf = None

for name, clf in classifiers.items():
    cv = cross_val_score(clf, X_meta_train, meta_train_y, cv=5, scoring='f1_macro')
    clf.fit(X_meta_train, meta_train_y)
    preds = clf.predict(X_meta_val)
    acc = (preds == val_y).mean()
    f1  = f1_score(val_y, preds, average='macro')
    
    marker = ''
    if f1 > best_f1:
        best_f1 = f1
        best_name = name
        best_clf = clf
        marker = ' <-- best'
    
    print(f'{name:<25} {cv.mean():>10.4f}   {acc:>10.4f} {f1:>10.4f}{marker}')

print(f'\nMelhor: {best_name} (F1={best_f1:.4f})')
meta_clf = best_clf

Classificador              CV F1-macro    Val Acc     Val F1
------------------------------------------------------------
Logistic Regression           0.9822       0.7200     0.6436 <-- best
Logistic Reg (C=10)           0.9825       0.7280     0.6578 <-- best
SVM (RBF)                     0.9708       0.7280     0.6560
SVM (Linear)                  0.9825       0.6960     0.6060
Random Forest                 0.9822       0.7120     0.6267
Gradient Boosting             0.9825       0.6880     0.5944
KNN (k=5)                     0.9880       0.7360     0.6560
KNN (k=3)                     0.9880       0.7280     0.6496
MLP                           0.9678       0.7200     0.6533

Melhor: Logistic Reg (C=10) (F1=0.6578)


## *avaliação no dataset-exemplos*

In [7]:
# predições do ensemble
ensemble_preds = meta_clf.predict(X_meta_val)

# predições individuais para comparação
roberta_preds = probs_roberta_val.argmax(axis=1)
deberta_preds = probs_deberta_val.argmax(axis=1)

# média simples para comparação
avg_probs = (probs_roberta_val + probs_deberta_val) / 2
avg_preds = avg_probs.argmax(axis=1)

print('=' * 60)
print('COMPARAÇÃO DE RESULTADOS (dataset-exemplos)')
print('=' * 60)

for name, preds in [('RoBERTa-base (sozinho)', roberta_preds),
                     ('DeBERTa-v3 (sozinho)',   deberta_preds),
                     ('Média simples',          avg_preds),
                     ('Stacking ensemble',      ensemble_preds)]:
    acc = (preds == val_y).mean()
    f1  = f1_score(val_y, preds, average='macro')
    print(f'\n{name}: accuracy={acc:.4f} | f1-macro={f1:.4f}')

COMPARAÇÃO DE RESULTADOS (dataset-exemplos)

RoBERTa-base (sozinho): accuracy=0.7760 | f1-macro=0.7186

DeBERTa-v3 (sozinho): accuracy=0.6960 | f1-macro=0.6060

Média simples: accuracy=0.7360 | f1-macro=0.6533

Stacking ensemble: accuracy=0.7280 | f1-macro=0.6578


In [8]:
print('\n' + '=' * 60)
print('CLASSIFICATION REPORT — Stacking Ensemble')
print('=' * 60)
print(classification_report(
    [ID2LABEL[y] for y in val_y],
    [ID2LABEL[p] for p in ensemble_preds],
    digits=3
))

print('=' * 60)
print('CLASSIFICATION REPORT — RoBERTa-base (referência)')
print('=' * 60)
print(classification_report(
    [ID2LABEL[y] for y in val_y],
    [ID2LABEL[p] for p in roberta_preds],
    digits=3
))


CLASSIFICATION REPORT — Stacking Ensemble
              precision    recall  f1-score   support

   anthropic      1.000     0.391     0.562        23
      google      0.789     0.938     0.857        16
       human      0.800     0.923     0.857        52
        meta      0.467     0.824     0.596        17
      openai      0.714     0.294     0.417        17

    accuracy                          0.728       125
   macro avg      0.754     0.674     0.658       125
weighted avg      0.778     0.728     0.707       125

CLASSIFICATION REPORT — RoBERTa-base (referência)
              precision    recall  f1-score   support

   anthropic      0.769     0.435     0.556        23
      google      0.636     0.875     0.737        16
       human      0.891     0.942     0.916        52
        meta      0.636     0.824     0.718        17
      openai      0.769     0.588     0.667        17

    accuracy                          0.776       125
   macro avg      0.740     0.733     

## *guardar ensemble*

In [9]:
with open('../models/ensemble_meta_clf.pkl', 'wb') as f:
    pickle.dump(meta_clf, f)

print('meta-classificador guardado em models/ensemble_meta_clf.pkl')
print('\npara inferência:')
print('  1. carregar RoBERTa + DeBERTa')
print('  2. extrair softmax de ambos')
print('  3. concatenar e passar pelo meta_clf')

meta-classificador guardado em models/ensemble_meta_clf.pkl

para inferência:
  1. carregar RoBERTa + DeBERTa
  2. extrair softmax de ambos
  3. concatenar e passar pelo meta_clf
